# Census Tract Analysis Demo

This notebook demonstrates how to aggregate rental data from zip codes to **census tracts** for more granular neighborhood analysis.

## Why Census Tracts?

- **More granular** than community areas (800+ tracts vs 77 communities in Chicago)
- **Standardized** across the entire US by the Census Bureau
- **Smaller areas** allow for more localized analysis
- **Better for equity analysis** - can identify disparities at block-level

## The Challenge: Zip ↔ Tract Mapping

Unlike community areas, **zip codes and census tracts have a many-to-many relationship**:
- A single zip code can overlap **multiple tracts**
- A single tract can span **multiple zip codes**
- We need **area-weighted aggregation** to handle overlaps properly

## The Goal

1. Loading census tract boundary data
2. Performing spatial joins with many-to-many relationships
3. Creating a zip-to-tract crosswalk
4. Aggregating data at the tract level
5. Comparing tract-level vs community-level analysis


## Step 1: Import the Pipeline Components


In [ ]:
from pipeline import (
    Pipeline,
    RentalDataLoader,
    TractAnalyzer,
    TractBoundariesLoader,
    ZipBoundariesLoader,
    ZipToTractProcessor,
)
from pipeline.config import PipelineConfig

print("✓ Pipeline system imported successfully!")

## Step 2: Get Census Tract Data

Before running the pipeline, you need to download census tract boundaries from the Box folder and place them in `data/`.


## Step 3: Create and Execute the Pipeline


In [ ]:
# Create the pipeline
config = PipelineConfig()
pipeline = Pipeline("Tract-Level Rental Analysis", config=config)
pipeline.load_config()

# Step 1: Load data sources
pipeline.register_component(RentalDataLoader())
pipeline.register_component(ZipBoundariesLoader())
pipeline.register_component(TractBoundariesLoader())

# Step 2: Perform spatial join (zip → tract)
pipeline.register_component(ZipToTractProcessor())

# Step 3: Analyze at tract level
pipeline.register_component(TractAnalyzer())

print("Pipeline created with 5 components")
print("\nPipeline workflow:")
for i, (_name, component) in enumerate(pipeline.components.items(), 1):
    print(f"{i}. {component.name} - {component.description}")

In [ ]:
# Execute the pipeline
print("Starting pipeline execution...")
print("=" * 60)

results = pipeline.execute()

print("\n" + "=" * 60)
print("Pipeline execution completed!")
print(f"\nGenerated {len(results)} results:")
for result in results:
    status = "✓ SUCCESS" if result.success else "✗ FAILED"
    print(f"{status}: {result.component_name} ({result.execution_time:.2f}s)")

## Step 4: Explore the Zip-to-Tract Crosswalk


In [ ]:
# Access the crosswalk from the pipeline context
crosswalk = pipeline.context.get("zip_to_tract_crosswalk")

if crosswalk is not None:
    print("ZIP-TO-TRACT CROSSWALK")
    print("=" * 60)
    print(f"Total mappings: {len(crosswalk)}")
    print(f"Unique zip codes: {crosswalk['zip_code'].nunique()}")
    print(f"Unique tracts: {crosswalk['tract_geoid'].nunique()}")

    # Show example: one zip code mapped to multiple tracts
    example_zip = crosswalk.groupby("zip_code").size().idxmax()
    example_mappings = crosswalk[crosswalk["zip_code"] == example_zip]

    print(f"\nExample: Zip code {example_zip} overlaps {len(example_mappings)} tracts:")
    print(
        example_mappings[["zip_code", "tract_geoid", "intersection_area"]]
        .head(10)
        .to_string(index=False)
    )
else:
    print("Crosswalk not found. Check if ZipToTractProcessor ran successfully.")

## Step 5: Aggregate Tracts to Community Areas

Now let's demonstrate aggregating tract-level data UP to community areas.


In [ ]:
from pipeline import CommunityBoundariesLoader, TractToCommunityProcessor

# Add community boundaries and aggregation to our existing pipeline
pipeline.register_component(CommunityBoundariesLoader())
pipeline.register_component(TractToCommunityProcessor())

# Run just the new components
new_results = []
for name in ["community_boundaries", "tract_to_community"]:
    component = pipeline.components[name]
    result = component.execute(pipeline.context)
    pipeline.context.update(result.data if hasattr(result, "data") else result)

print("✓ Aggregated tract data to community areas!")

In [ ]:
# Access community-level aggregated data
community_data = pipeline.context.get("community_rental_data")

if community_data is not None:
    print("COMMUNITY-LEVEL AGGREGATED DATA")
    print("=" * 60)
    print(f"Total community areas: {len(community_data)}")
    print(f"Communities with data: {community_data['avg_rental_price'].notna().sum()}")

    # Compare with tract-level
    tract_data = pipeline.context["tract_rental_data"]
    tract_with_data = tract_data["avg_rental_price"].notna().sum()

    print("\nGranularity comparison:")
    print(f"  Census tracts: {tract_with_data} areas with data")
    print(
        f"  Community areas: {community_data['avg_rental_price'].notna().sum()} areas with data"
    )
    print(
        f"  Ratio: {tract_with_data / community_data['avg_rental_price'].notna().sum():.1f}x more granular"
    )

    # Show top communities
    print("\nTop 5 Most Expensive Community Areas:")
    top_communities = community_data.nlargest(5, "avg_rental_price")[
        ["community_name", "avg_rental_price", "tract_count"]
    ]
    print(top_communities.to_string(index=False))

## Summary

### What Was Accomplished

1. ✓ Loaded census tract boundary data
2. ✓ Performed spatial join from zip codes to tracts  
3. ✓ Created a zip-to-tract crosswalk with area weights
4. ✓ Aggregated rental data at the tract level
5. ✓ **Aggregated tract data up to community areas** (cleaner than zip→community!)
6. ✓ Analyzed patterns at multiple geographic levels

### Key Insights

- **Hierarchical aggregation**: Zip → Tract → Community is cleaner than Zip → Community
- **Many-to-many mapping**: A single zip can span multiple tracts and vice versa
- **Area weighting is critical**: We weight prices by intersection area for accuracy
- **Tracts as base unit**: Census tracts are the ideal base for multi-level analysis
- **Crosswalk utility**: The zip-to-tract crosswalk can be reused for other datasets


## Next Steps

### 1. Run the Example Notebook
```bash
# In your dev container, open notebooks/tract_analysis_demo.ipynb
```

### 2. Export Results
```python
# Save crosswalk for reuse
crosswalk.to_csv('/project/data/zip_to_tract_crosswalk.csv', index=False)

# Save tract data
tract_data.to_csv('/project/data/tract_rental_prices.csv', index=False)

# Save as GeoJSON for mapping
tract_data.to_file('/project/data/tract_rental_prices.geojson', driver='GeoJSON')
```

### 3. Join with Census Data
```python
# Get demographic data from Census
import cenpy
acs = cenpy.products.ACS(2020)
demographics = acs.from_county('Cook County, IL', level='tract', variables=['B19013_001E'])  # Median income

# Join with rental data
enriched = tract_data.merge(demographics, left_on='tract_geoid', right_index=True)
```